# Differential gene expression - Cell type (sub clusters)
Assesing if there is a difference if I run using only unique donors
Which means that donors that were used in different datasets, I will only keep the donor in one dataset

In [ ]:
# eval "$(conda shell.bash hook)"
# conda init
# conda activate /work/islet_cartography_scrna/scrna_cartography_dreampy
# python -m ipykernel install --user --name scrna_cartography_dreampy --display-name "dreampy"

In [ ]:
# Path and system utilities
import os                    # Operating system interface
import sys                   # System-specific parameters and functions
import glob                  # File pattern matching
from pathlib import Path     # Object-oriented filesystem paths
from pyhere import here      # Reproducible project paths
import gc

# Single-cell data handling
import anndata as ad            # Core data structure for single-cell data
import scanpy as sc

import matplotlib.pyplot as plt  # Plotting interface

# dream
from pydeseq2.dds import DeseqDataSet
from pydeseq2.default_inference import DefaultInference
from pydeseq2.ds import DeseqStats
import dreampy as dp

# Parallel processing
from joblib import Parallel, delayed, parallel_backend

# dataframes
import pandas as pd
import numpy as np
from collections import defaultdict

# Custom modules and functions
sys.path.append(str(here('scripts/misc')))  # Add custom script path to system
import misc as mi
import diff_genes as dg

In [ ]:
# Paths
base_dir = str(here('data/annotate/'))
plot_dir = os.path.join(base_dir, 'plot') 
files_dir = os.path.join(base_dir, 'files') 
diffg_dir = os.path.join(base_dir, 'deseq_onevsother_unique') 
tmp_dir = os.path.join(base_dir, 'tmp') 

anndata_dir = str(here('data/anndata/'))

mi.create_directories(os.path.join(base_dir, 'deseq_onevsother_unique'))
mi.create_directories(os.path.join(base_dir, 'tmp'))

In [ ]:
adata = ad.read_h5ad(os.path.join(anndata_dir, "AH_combined.h5ad"))

## UMAP

In [ ]:
sc.pl.umap(
    adata,
    color='cell_type',
    legend_fontsize=5,
    size=1,
    show=False  
)

ax = plt.gca()

# Remove border
for spine in ax.spines.values():
    spine.set_visible(False)

# Remove ticks
ax.set_xticks([])
ax.set_yticks([])

# Rasterize points
for coll in ax.collections:
    coll.set_rasterized(True)

plt.savefig(
    os.path.join(plot_dir, "umap_cell_type.pdf"),
    bbox_inches="tight", 
    dpi=300
)
plt.close()

## Meta data

In [ ]:
adata.obs.to_csv(os.path.join(files_dir, 'obs.csv'), index=True, index_label = 'barcode')

## Differential gene expression

In [ ]:
# Setup -----------------------------------------------------------------------------
anno_key   = "cell_type"
sample_key = "ic_id_platform_adjusted_sample"
donor_key  = "ic_id_donor_overall"
dataset_key  = "ic_id_dataset"
disease_key = "disease_harmonized"
target_celltypes = adata.obs[anno_key].unique()
inference = DefaultInference(n_cpus=-1)

#### % of cells express

In [ ]:
# Calculate percent of genes expressed
perc_all = dg.compute_pct_expressing(adata, anno_key, workers = -1)
perc_all.to_csv(os.path.join(diffg_dir, f"percentage_of_genes_expressed.csv"), index=True, index_label = "gene_symbol")

#### Mean expression

In [ ]:
mean_cell_type = sc.get.aggregate(
    adata, 
    by=anno_key,  
    func="mean"           
)

mean_df = pd.DataFrame(
    mean_cell_type.layers['mean'],
    columns=mean_cell_type.var_names,
    index=mean_cell_type.obs_names
)

mean_df.transpose().to_csv(os.path.join(diffg_dir, f"mean_genes_expressed.csv"), index=True, index_label = "gene_symbol")

#### Identify duplicated donors

In [ ]:
# Find donors appearing in multiple datasets
donor_dataset_counts = adata.obs.groupby(donor_key)[dataset_key].nunique()
multi_dataset_donors = donor_dataset_counts[donor_dataset_counts > 1].index

print(f"Found {len(multi_dataset_donors)} donors in multiple datasets")

# Check which multi-dataset donors have 10x available
is_multi = adata.obs[donor_key].isin(multi_dataset_donors)
is_10x = adata.obs['library_prep'].str.contains('10x', case=False)

multi_with_10x = adata.obs[is_multi & is_10x][donor_key].nunique()
multi_without_10x = len(multi_dataset_donors) - multi_with_10x

print(f"  - {multi_with_10x} donors have 10x available (will prioritize)")
print(f"  - {multi_without_10x} donors only in smart-seq (will not keep those)")

# Keep: single-dataset donors + 10x versions of multi-dataset donors
keep = ~is_multi | is_10x
adata = adata[keep].copy()

print(f"Removed {(~keep).sum()} observations")

# Verify filtering worked
remaining_multi = adata.obs.groupby(donor_key)[dataset_key].nunique()
n_remaining_multi = (remaining_multi > 1).sum()
assert n_remaining_multi == 0, f"Error: {n_remaining_multi} donors still appear in multiple datasets!"
print("Filtering successful: all donors now in single dataset")
print("Number of donors: ", len(remaining_multi))

In [ ]:
del is_multi, is_10x, multi_with_10x, multi_without_10x, keep, remaining_multi, n_remaining_multi
gc.collect()

#### Diff genes between specific clusters - per dataset

In [ ]:
# Paths
diffg_dir = os.path.join(base_dir, 'deseq_onevsother_unique_between') 
mi.create_directories(os.path.join(base_dir, 'deseq_onevsother_unique_between'))

In [ ]:
comparison_pairs = [
    ("endothelial", "pericyte"),
    ("acinar", "acinar_reg_plus"),
    ("ductal", "ductal_mucin")
]

celltypes_needed = list({ct for pair in comparison_pairs for ct in pair})
adata_sub = adata[adata.obs[anno_key].isin(celltypes_needed)].copy()

for cluster_id, ref_id in comparison_pairs:

    comp = f"{cluster_id}_vs_{ref_id}"
    print(f"\n=== {comp} ===")
    meta_results = []

    ad_pair = adata_sub[adata_sub.obs[anno_key].isin([cluster_id, ref_id])].copy()
    ad_pair.obs[comp] = (ad_pair.obs[anno_key] == cluster_id).map({True: cluster_id, False: ref_id})
    ad_pair.obs["assay"] = "my_assay"

    for dataset in ad_pair.obs[dataset_key].unique():

        ad_ds = ad_pair[ad_pair.obs[dataset_key] == dataset].copy()

        if not {cluster_id, ref_id}.issubset(ad_ds.obs[comp].unique()):
            print(f"  Skipping {dataset}: missing cell type(s)")
            continue

        try:
            pb = dp.aggregate_pseudobulk(
                ad_ds, layer="counts", groupby=["assay", donor_key, comp]
            )

            # Find the lowest min_cells threshold that gives ≥3 replicates for BOTH groups
            min_cells_used = None
            pb_filtered = None
            for min_cells in [50, 20, 10, 5, 3, 2, 1]:
                pb_try = dp.filter_samples(pb, min_cells=min_cells, min_samples=3)
                counts = pb_try.obs.groupby(comp)["assay"].count()
                if counts.get(cluster_id, 0) >= 3 and counts.get(ref_id, 0) >= 3:
                    pb_filtered = pb_try
                    min_cells_used = min_cells
                    break

            if pb_filtered is None:
                raise ValueError("Could not get ≥3 replicates for both groups at any threshold")

            counts_df = pd.DataFrame(
                pb_filtered.X.toarray(),
                columns=pb_filtered.var_names,
                index=pb_filtered.obs_names,
            )
            metadata_df = pb_filtered.obs[[donor_key, comp]].copy()
            assert counts_df.index.equals(metadata_df.index)

            dds = DeseqDataSet(
                counts=counts_df,
                metadata=metadata_df,
                design=f"~ {donor_key} + {comp}",
                inference=inference,
            )
            dds.deseq2()

            ds = DeseqStats(
                dds, contrast=(comp, cluster_id, ref_id), inference=inference, quiet=True
            )
            ds.run_wald_test()
            ds.summary()

            donors_used = pb_filtered.obs[donor_key].unique().tolist()
            perc = dg.compute_pct_expressing(
                ad_ds[ad_ds.obs[donor_key].isin(donors_used)], anno_key
            )

            n_donor = pd.concat([
                pb_filtered.obs[disease_key].value_counts().to_frame().T,
                pb_filtered.obs[comp].value_counts().to_frame().T,
            ], axis=1)
            perc[n_donor.columns.tolist()] = n_donor.iloc[0]

            results = ds.results_df.copy()
            results["target"] = cluster_id
            results["reference"] = ref_id
            results["comparison"] = comp
            results["n_donors"]   = dds.shape[0]
            results["dataset"]    = dataset
            results["min_cells"]  = min_cells_used
            results = results.join(perc, how="left")

            meta_results.append(results)
            results.to_csv(
                os.path.join(tmp_dir, f"{comp}_{dataset}.csv"),
                index=True, index_label="gene_symbol",
            )
    
        except Exception as e:
            print(f"  Skipping {dataset}: {e}")
    
    # Combine all results for this cluster
    if len(meta_results) > 0:
        meta_df = pd.concat(meta_results, ignore_index=False)
        meta_df.to_csv(
            os.path.join(diffg_dir, f"{cluster_id}.csv"), 
            index=True, 
            index_label="gene_symbol"
        )
    else:
        print(f"Skipping entire cluster {cluster_id} - all datasets failed")